In [40]:
import requests
import json

class WikidataFormulizer:
    def __init__(self):
        pass  # No ontology yet, just fetching relationships

    def query_wikidata_relationships(self, wikidata_id):
        """
        Query Wikidata for only essential relationships (instance of, subclass of).
        """
        query = f"""
        SELECT ?property ?propertyLabel ?value ?valueLabel WHERE {{
          wd:{wikidata_id} ?property ?value .
          VALUES ?property {{ wd:P31 wd:P279 }}  # Only allow essential properties
          SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
        }}
        """
        url = "https://query.wikidata.org/sparql"
        headers = {"Accept": "application/sparql-results+json"}
        response = requests.get(url, params={"query": query}, headers=headers)

        if response.status_code == 200:
            results = response.json()["results"]["bindings"]
            relationships = []
            for result in results:
                relationships.append({
                    "subject": wikidata_id,
                    "predicate": result["propertyLabel"]["value"],
                    "object": result["valueLabel"]["value"]
                })
            return relationships
        else:
            print(f"⚠️ Wikidata query failed for {wikidata_id}")
            return []

    def process_annotation(self, annotation):
        """
        Extract relationships from Wikidata for each entity in the annotation JSON.
        """
        print(f"\n🔍 Processing sentence: {annotation['original_sentence']}")
        for entity in annotation["annotations"]:
            entity_label = entity["label"]
            wikidata_id = entity["wikidata_id"]

            print(f"📡 Fetching relationships for: {entity_label} ({wikidata_id})")
            relationships = self.query_wikidata_relationships(wikidata_id)

            if relationships:
                for rel in relationships:
                    print(f"🔗 {rel['subject']} → {rel['predicate']} → {rel['object']}")
            else:
                print(f"⚠️ No useful relationships found for {entity_label}")

    def process_json_annotations(self, json_annotations):
        """
        Process multiple entities and fetch their relationships.
        """
        for annotation in json_annotations:
            self.process_annotation(annotation)

# Example JSON input from annotator
json_data = """
[
    {
        "original_sentence": "Python is a widely used programming language.",
        "annotations": [
            {
                "entity": "Python",
                "wikidata_id": "Q28865",
                "label": "Python",
                "description": "general-purpose programming language",
                "wikidata_url": "https://www.wikidata.org/wiki/Q28865",
                "relevance_score": 1.0
            }
        ]
    }
]
"""

# Convert JSON string to Python object
json_annotations = json.loads(json_data)

# Initialize the Wikidata Formulizer (No OWL yet, just testing relationships)
wikidata_formulizer = WikidataFormulizer()

# Process Wikidata annotations
wikidata_formulizer.process_json_annotations(json_annotations)



🔍 Processing sentence: Python is a widely used programming language.
📡 Fetching relationships for: Python (Q28865)
⚠️ No useful relationships found for Python


In [39]:
def query_wikidata_relationships(self, wikidata_id):
    """
    Query Wikidata for more relationships, not just P31 and P279.
    """
    query = f"""
    SELECT ?property ?propertyLabel ?value ?valueLabel WHERE {{
      wd:{wikidata_id} ?property ?value .
      VALUES ?property {{ wd:P31 wd:P279 wd:P361 wd:P527 wd:P366 }}  # More properties
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    url = "https://query.wikidata.org/sparql"
    headers = {"Accept": "application/sparql-results+json"}
    response = requests.get(url, params={"query": query}, headers=headers)

    if response.status_code == 200:
        results = response.json()["results"]["bindings"]
        relationships = []
        for result in results:
            relationships.append({
                "subject": wikidata_id,
                "predicate": result["propertyLabel"]["value"],
                "object": result["valueLabel"]["value"]
            })
        return relationships
    else:
        print(f"⚠️ Wikidata query failed for {wikidata_id}")
        return []


In [44]:
query_wikidata_relationships(wikidata_id='Q28865')

TypeError: query_wikidata_relationships() missing 1 required positional argument: 'self'

In [47]:
def query_wikidata_relationships(self, wikidata_id):
    """
    Query Wikidata for relationships WITHOUT filtering properties.
    """
    query = f"""
    SELECT ?property ?propertyLabel ?value ?valueLabel WHERE {{
      wd:{wikidata_id} ?property ?value .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    LIMIT 50  # Limit to avoid excessive data
    """
    url = "https://query.wikidata.org/sparql"
    headers = {"Accept": "application/sparql-results+json"}
    response = requests.get(url, params={"query": query}, headers=headers)

    if response.status_code == 200:
        data = response.json()
        print(json.dumps(data, indent=4))  # Print full response for debugging
        return data["results"]["bindings"]  # Return raw data
    else:
        print(f"⚠️ Wikidata query failed for {wikidata_id}")
        return []


In [48]:
formulizer = WikidataFormulizer()
relationships = formulizer.query_wikidata_relationships("Q28865")  # Python


In [49]:
from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS, OWL

# Define namespaces
WD = Namespace("http://www.wikidata.org/entity/")
EX = Namespace("http://example.org/ontology/")

def formulizer(annotated_sentence):
    # Create an RDF graph
    g = Graph()

    # Bind namespaces
    g.bind("wd", WD)
    g.bind("ex", EX)
    g.bind("owl", OWL)
    g.bind("rdfs", RDFS)

    # Extract the original sentence and annotations
    original_sentence = annotated_sentence["original_sentence"]
    annotations = annotated_sentence["annotations"]

    # Iterate through annotations and create OWL axioms
    for annotation in annotations:
        entity_uri = URIRef(annotation["wikidata_url"])  # Use Wikidata URL as the entity URI
        entity_label = Literal(annotation["label"])
        entity_description = Literal(annotation["description"])

        # Add entity as an individual of a class
        g.add((entity_uri, RDF.type, EX.ProgrammingLanguage))
        g.add((entity_uri, RDFS.label, entity_label))
        g.add((entity_uri, RDFS.comment, entity_description))

    return g

# Example usage
annotated_sentence = {
    "original_sentence": "Python is a widely used programming language.",
    "annotations": [
        {
            "entity": "Python",
            "wikidata_id": "Q28865",
            "label": "Python",
            "description": "general-purpose programming language",
            "wikidata_url": "https://www.wikidata.org/wiki/Q28865",
            "relevance_score": 1.0
        }
    ]
}

# Generate OWL representation
owl_graph = formulizer(annotated_sentence)

# Serialize the OWL graph to Turtle format
print(owl_graph.serialize(format="turtle"))

@prefix ex: <http://example.org/ontology/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

<https://www.wikidata.org/wiki/Q28865> a ex:ProgrammingLanguage ;
    rdfs:label "Python" ;
    rdfs:comment "general-purpose programming language" .




In [1]:
import json

def formulate_statement(annotated_output):
    """
    Convert annotated sentences into a structured formal representation.
    Uses Wikidata properties for relationships and ChEBI IDs for chemicals.
    """
    data = json.loads(annotated_output)
    statements = []

    for relation in data.get("relationships", []):
        subj = relation["subject"]
        pred = relation["predicate"]
        obj = relation["object"]

        # Try to map the subject and object to their respective IDs
        subj_id = None
        obj_id = None

        for entity in data.get("annotations", []):
            if entity["entity"].lower() == subj.lower():
                subj_id = entity.get("chebi_id", entity.get("wikidata_id", subj))
            if entity["entity"].lower() == obj.lower():
                obj_id = entity.get("chebi_id", entity.get("wikidata_id", obj))

        # Use Wikidata property if available
        prop_id = relation.get("property_id", pred)

        # Generate RDF triple
        rdf_triple = f"<{subj_id}> <{prop_id}> <{obj_id}>."
        statements.append(rdf_triple)

    return {
        "original_sentence": data["original_sentence"],
        "formulated_statements": statements
    }

# Example Usage
if __name__ == "__main__":
    # Simulated Annotator Output
    annotator_output = """
    {
        "original_sentence": "Aspirin inhibits cyclooxygenase enzymes.",
        "annotations": [
            {
                "entity": "Aspirin",
                "chebi_id": "CHEBI:15365",
                "label": "Aspirin",
                "description": "A salicylic acid derivative used as an anti-inflammatory drug.",
                "chebi_url": "https://www.ebi.ac.uk/chebi/searchId.do?chebiId=CHEBI:15365"
            },
            {
                "entity": "cyclooxygenase enzymes",
                "chebi_id": "CHEBI:37926",
                "label": "Cyclooxygenase",
                "description": "An enzyme that is the target of NSAIDs.",
                "chebi_url": "https://www.ebi.ac.uk/chebi/searchId.do?chebiId=CHEBI:37926"
            }
        ],
        "relationships": [
            {
                "subject": "Aspirin",
                "predicate": "inhibit",
                "object": "cyclooxygenase enzymes",
                "property_id": "wdt:P2157"
            }
        ]
    }
    """

    # Generate formulized output
    output = formulate_statement(annotator_output)
    print(json.dumps(output, indent=4))


{
    "original_sentence": "Aspirin inhibits cyclooxygenase enzymes.",
    "formulated_statements": [
        "<CHEBI:15365> <wdt:P2157> <CHEBI:37926>."
    ]
}
